In [1]:
import torchaudio
torchaudio.set_audio_backend("soundfile")

C:\Users\Andrei\AppData\Local\Temp\ipykernel_30244\2654335069.py:2: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")


In [2]:
import torch
import torchaudio
from tqdm import tqdm

# -------- CONFIG --------
SAMPLE_RATE = 16000

# MFCC (lightweight)
mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=SAMPLE_RATE,
    n_mfcc=40,  # standard choice
    melkwargs={
        "n_fft": 400,
        "hop_length": 160,
        "n_mels": 64
    }
)

# Log-Mel Spectrogram (heavy model)
mel_spec_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=400,
    hop_length=160,
    n_mels=128
)

amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\torchaudio\functional\functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(


In [3]:
def extract_features(waveform, mode="mfcc"):
    """
    waveform: Tensor [1, T]
    mode: 'mfcc' or 'mel'
    """
    
    if mode == "mfcc":
        features = mfcc_transform(waveform)  # [1, n_mfcc, time]
        
    elif mode == "mel":
        features = mel_spec_transform(waveform)  # [1, n_mels, time]
        features = amplitude_to_db(features)
        
    else:
        raise ValueError("Unknown mode")

    return features.squeeze(0)  # [feat_dim, time]

In [4]:
MAX_LEN = SAMPLE_RATE * 6  # 6 seconds


def fix_length(waveform):
    """
    waveform: [1, T]
    """
    T = waveform.shape[1]

    if T > MAX_LEN:
        # truncate
        waveform = waveform[:, :MAX_LEN]
    else:
        # pad
        pad_size = MAX_LEN - T
        waveform = torch.nn.functional.pad(waveform, (0, pad_size))

    return waveform

In [5]:
import librosa
import torch

def load_audio(path):
    # librosa loads reliably on Windows
    y, sr = librosa.load(path, sr=SAMPLE_RATE)

    # convert to torch
    waveform = torch.tensor(y, dtype=torch.float32).unsqueeze(0)

    waveform = fix_length(waveform)

    return waveform

In [6]:
import pandas as pd
from torch.utils.data import Dataset
from pathlib import Path

# -------- PATHS --------
DATA_ROOT = Path("../Data/MELD.Audio")

# label mapping (important)
EMO_MAP = {
    "joy": "happiness"
}

# final label set (fixed order!)
LABELS = [
    "anger",
    "disgust",
    "fear",
    "happiness",
    "sadness",
    "surprise",
    "neutral"
]

label2idx = {l: i for i, l in enumerate(LABELS)}

In [7]:
class MELDAudioDataset(Dataset):
    def __init__(self, split="train", mode="mfcc"):
        self.split = split
        self.mode = mode

        # load CSV
        if split == "train":
            csv_path = "../Data/MELD.Raw/train/train_sent_emo.csv"
        elif split == "dev":
            csv_path = "../Data/MELD.Raw/dev_sent_emo.csv"
        else:
            csv_path = "../Data/MELD.Raw/test_sent_emo.csv"

        df = pd.read_csv(csv_path)

        # normalize labels
        df["Emotion"] = df["Emotion"].str.lower()
        df["Emotion"] = df["Emotion"].replace(EMO_MAP)

        self.audio_dir = DATA_ROOT / split

        # filter missing files
        valid_rows = []
        for _, row in df.iterrows():
            dia = row["Dialogue_ID"]
            utt = row["Utterance_ID"]

            filename = f"dia{dia}_utt{utt}.wav"
            path = self.audio_dir / filename

            if path.exists():
                valid_rows.append(row)

        self.df = pd.DataFrame(valid_rows).reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        dia = row["Dialogue_ID"]
        utt = row["Utterance_ID"]

        filename = f"dia{dia}_utt{utt}.wav"
        path = self.audio_dir / filename

        waveform = load_audio(path)
        features = extract_features(waveform, self.mode)

        label = label2idx[row["Emotion"]]

        return features, label

In [8]:
train_ds = MELDAudioDataset("train")
print(len(train_ds))
x, y = train_ds[0]

9988


e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
torchaudio.get_audio_backend()

C:\Users\Andrei\AppData\Local\Temp\ipykernel_30244\1746710290.py:1: UserWarning: torchaudio._backend.get_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.get_audio_backend()


In [10]:
ds = MELDAudioDataset(split="train", mode="mfcc")

x, y = ds[0]

print(x.shape)  # [feat_dim, time]
print(y)        # int

torch.Size([40, 601])
6


Small CNN on MFCC (fast, low params, decent performance)

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class AudioCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(0.3)

        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(24000, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # x: [B, feat, time] → add channel
        x = x.unsqueeze(1)  # [B, 1, F, T]

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [12]:
model = AudioCNN()

x = torch.randn(1, 40, 601)
out = model.conv1(x.unsqueeze(1))
out = model.pool(out)
out = model.pool(model.conv2(out))
out = model.pool(model.conv3(out))

print(out.shape)

torch.Size([1, 64, 5, 75])


In [13]:
model = AudioCNN()

x = torch.randn(2, 40, 601)  # batch of 2
out = model(x)

print(out.shape)  # should be [2, 7]

torch.Size([2, 7])


## Data loader

In [14]:
from torch.utils.data import DataLoader

train_ds = MELDAudioDataset(split="train", mode="mfcc")
dev_ds   = MELDAudioDataset(split="dev", mode="mfcc")

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
dev_loader   = DataLoader(dev_ds, batch_size=32, shuffle=False)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [16]:
model = AudioCNN().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2+cu118
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Training loop

In [17]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        out = model(x)
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

Evaluation

In [18]:
def evaluate(model, loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            out = model(x)
            preds = out.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

Train

In [82]:
EPOCHS = 10

best_acc = 0

for epoch in tqdm(range(EPOCHS)):
    train_loss = train_one_epoch(model, train_loader)
    val_acc = evaluate(model, dev_loader)

    print(f"Epoch {epoch+1}: loss={train_loss:.4f}, val_acc={val_acc:.4f}")

    # save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "../Models/Audio_Models/best_audio_model.pth")

 10%|█         | 1/10 [01:36<14:30, 96.67s/it]

Epoch 1: loss=1.5339, val_acc=0.4233


 20%|██        | 2/10 [03:14<12:57, 97.22s/it]

Epoch 2: loss=1.5183, val_acc=0.4341


 30%|███       | 3/10 [04:51<11:21, 97.29s/it]

...

Epoch 8: loss=0.8637, val_acc=0.3709


 90%|█████████ | 9/10 [14:31<01:37, 97.21s/it]

Epoch 9: loss=0.6877, val_acc=0.3682


100%|██████████| 10/10 [16:09<00:00, 96.95s/it]

Epoch 10: loss=0.5322, val_acc=0.3601


In [19]:
train_ds = MELDAudioDataset(split="train", mode="mel")
dev_ds   = MELDAudioDataset(split="dev", mode="mel")

In [20]:
import torch
from collections import Counter

train_labels = [label for _, label in train_ds]
class_counts = Counter(train_labels)

print("Class counts:", class_counts)

Class counts: Counter({6: 4709, 3: 1743, 5: 1205, 0: 1109, 4: 683, 1: 271, 2: 268})


In [21]:
num_classes = 7

counts = torch.tensor(
    [class_counts[i] for i in range(num_classes)],
    dtype=torch.float32
)

# inverse frequency
class_weights = 1.0 / torch.sqrt(counts)

# normalize so average weight is ~1
class_weights = class_weights / class_weights.mean()

print("Class weights:", class_weights)

Class weights: tensor([0.8165, 1.6516, 1.6608, 0.6512, 1.0404, 0.7833, 0.3962])


In [22]:
class_weights = class_weights.to(device)

In [34]:
from torch.utils.data import WeightedRandomSampler

sample_weights = [1.0 / (class_counts[label] ** 0.5) for _, label in train_ds]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

In [35]:
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    sampler=sampler,
    num_workers=0
)

dev_loader = DataLoader(
    dev_ds,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

In [36]:
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

In [37]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def evaluate_full(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            out = model(x)
            preds = out.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    report = classification_report(
        all_labels,
        all_preds,
        target_names=LABELS,
        digits=4
    )

    return acc, macro_f1, report

In [38]:
class AudioCNNHeavy(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)

        self.pool = nn.MaxPool2d(2)

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))

        # ✅ reduce to fixed size
        x = self.global_pool(x)      # [B, 256, 1, 1]
        x = x.view(x.size(0), -1)    # [B, 256]

        x = self.dropout(x)
        x = self.fc(x)

        return x

In [39]:
model = AudioCNNHeavy()

x = torch.randn(1, 128, 601)
out = model.conv1(x.unsqueeze(1))
out = model.pool(out)
out = model.pool(model.conv2(out))
out = model.pool(model.conv3(out))
out = model.pool(model.conv4(out))

print(out.shape)

torch.Size([1, 256, 8, 37])


In [40]:
model = AudioCNNHeavy()
x = torch.randn(2, 128, 601)

out = model(x)
print(out.shape)  # should be [2, 7]

torch.Size([2, 7])


In [41]:
model = AudioCNNHeavy().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)  # lower LR
criterion = nn.CrossEntropyLoss()

In [153]:
%pip uninstall numpy -y
%pip install numpy==1.26.4

Found existing installation: numpy 2.3.5
Uninstalling numpy-2.3.5:
  Successfully uninstalled numpy-2.3.5
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.


  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
torchvision 0.22.1+cu118 requires torch==2.7.1+cu118, but you have torch 2.2.2+cu118 which is incompatible.


In [42]:
best_f1 = 0.0

for epoch in range(100):
    model.train()
    total_loss = 0.0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        out = model(x)
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    val_acc, val_macro_f1, val_report = evaluate_full(model, dev_loader, device)
    print(f"Epoch {epoch+1}: loss={train_loss:.4f}, val_acc={val_acc:.4f}, val_macro_f1={val_macro_f1:.4f}")

    if val_macro_f1 > best_f1:
        best_f1 = val_macro_f1
        torch.save(model.state_dict(), "best_audio_weighted.pth")
        print("Saved best model.")

print("\nBest macro F1:", best_f1)

e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contr

Epoch 1: loss=1.8415, val_acc=0.4188, val_macro_f1=0.0908
Saved best model.


e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contr

Epoch 2: loss=1.8358, val_acc=0.4106, val_macro_f1=0.0956
Saved best model.


e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contr

...

e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contr

Epoch 31: loss=1.6878, val_acc=0.4404, val_macro_f1=0.1970
Epoch 32: loss=1.6946, val_acc=0.4486, val_macro_f1=0.1949
Epoch 33: loss=1.6636, val_acc=0.4440, val_macro_f1=0.2094
Saved best model.
Epoch 34: loss=1.6562, val_acc=0.4404, val_macro_f1=0.2078
Epoch 35: loss=1.6370, val_acc=0.4368, val_macro_f1=0.1904
Epoch 36: loss=1.6344, val_acc=0.4413, val_macro_f1=0.1921
Epoch 37: loss=1.6197, val_acc=0.4477, val_macro_f1=0.1858
Epoch 38: loss=1.6057, val_acc=0.3709, val_macro_f1=0.2127
Saved best model.


e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contr

Epoch 39: loss=1.5918, val_acc=0.3953, val_macro_f1=0.1937
Epoch 40: loss=1.5673, val_acc=0.4395, val_macro_f1=0.2198
Saved best model.
Epoch 41: loss=1.5570, val_acc=0.4323, val_macro_f1=0.2157
Epoch 42: loss=1.5304, val_acc=0.4531, val_macro_f1=0.1930
Epoch 43: loss=1.5178, val_acc=0.4305, val_macro_f1=0.2088


KeyboardInterrupt: 

In [33]:
print(val_report)

              precision    recall  f1-score   support

       anger     0.5122    0.2745    0.3574       153
     disgust     0.0000    0.0000    0.0000        22
        fear     0.0625    0.0250    0.0357        40
   happiness     0.2200    0.2025    0.2109       163
     sadness     0.1143    0.0360    0.0548       111
    surprise     0.2564    0.1333    0.1754       150
     neutral     0.4906    0.7804    0.6025       469

    accuracy                         0.4206      1108
   macro avg     0.2366    0.2074    0.2052      1108
weighted avg     0.3592    0.4206    0.3659      1108

